# 1.0 — Non-Instruction Fine-Tuning on Base Model (Domain Adaptation)

## Purpose
This notebook performs the **first stage** of a three-step training pipeline for a BPMN-aware language model. The goal is to adapt the base `TinyLlama-1.1B` model to the vocabulary, structure, and semantics of the **BPMN 2.0 specification** using raw (non-instruction) text extracted from the official specification PDF.

## What This Notebook Does
1. **Extracts** structured text from the official BPMN 2.0 PDF specification using font-size and layout cues.
2. **Chunks** the extracted text into semantically meaningful, token-bounded segments (100–512 tokens each).
3. **Fine-tunes** TinyLlama with **LoRA (Low-Rank Adaptation)** adapters on the chunked text using a standard causal language modelling objective.
4. **Tests** the resulting domain-adapted model with a sample BPMN prompt.

## Output
A LoRA adapter checkpoint saved to `./tinyllama-lora/`, which is used as the starting point for instruction fine-tuning in notebook `2.0`.

## Step 1 — Install Dependencies

Install the required Python packages:
- **`peft`** — Parameter-Efficient Fine-Tuning (provides LoRA)
- **`bitsandbytes`** — 8-bit quantisation support for memory-efficient training
- **`transformers`** — Hugging Face model hub, tokenizers, and training utilities
- **`accelerate`** — Backend for distributed and mixed-precision training
- **`trl`** — Transformer Reinforcement Learning (used in later training stages)
- **`PyMuPDF`** — PDF text extraction library (imported as `fitz`)

In [1]:
#install dependecnies
!pip install -U peft bitsandbytes transformers accelerate
!pip install -U trl
!pip install PyMuPDF

  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   ---------------------------------------- 557.0/557.0 kB 9.5 MB/s  0:00:00
   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   ----- ---------------------------------- 7.1/55.4 MB 33.6 MB/s eta 0:00:02
   ----------- ---------------------------- 16.0/55.4 MB 37.3 MB/s eta 0:00:02
   ----------------- ---------------------- 24.6/55.4 MB 40.0 MB/s eta 0:00:01
   --------------------- ------------------ 29.1/55.4 MB 36.2 MB/s eta 0:00:01
   -------------------------- ------------- 36.2/55.4 MB 33.8 MB/s eta 0:00:01
   -------------------------------- ------- 45.1/55.4 MB 35.4 MB/s eta 0:00:01
   -------------------------------------- - 53.5/55.4 MB 36.2 MB/s eta 0:00:01
   ---------------------------------------- 55.4/55.4 MB 33.6 MB/s  0:00:01
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   -----------


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/528.8 kB ? eta -:--:--
   ---------------------------------------- 528.8/528.8 kB 8.6 MB/s  0:00:00



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Extract Domain-Specific Text from the BPMN Specification PDF

The training data comes from the **official BPMN 2.0 specification** (`formal-11-01-03.pdf`).

The `extract_text_from_pdf` function uses **block-level layout analysis** via PyMuPDF (`fitz`) to extract text spans along with their font size and bold/italic flags. This metadata is critical because:
- **Font size** helps distinguish headings from body text.
- **Bold flags** identify section headers and important terms.
- Extraction starts at **page 29** (zero-based index 28) to skip the table of contents and front matter.

The **dominant body font size** is detected automatically by finding the most frequent font size across all spans, which is then used by the chunker to identify headers.

In [31]:
import fitz
import re

def extract_text_from_pdf(pdf_path, start_page=0):
    """
    Extract text using block-level structure to preserve layout context.
    Returns a list of (page_num, block_text, font_size) tuples for smarter chunking.
    """
    blocks = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc[start_page:], start=start_page + 1):
            for block in page.get_text("dict")["blocks"]:
                if block["type"] != 0:  # skip non-text blocks (images, etc.)
                    continue
                for line in block["lines"]:
                    for span in line["spans"]:
                        text = span["text"].strip()
                        if text:
                            blocks.append({
                                "page": page_num,
                                "text": text,
                                "size": round(span["size"], 1),
                                "flags": span["flags"]  # bold=1, italic=2
                            })
    return blocks

# Extract text starting from page 29 (0-based index = 28)
pdf_blocks = extract_text_from_pdf("./data/formal-11-01-03.pdf", start_page=28)

# Detect dominant body font size (most frequent)
from collections import Counter
size_counts = Counter(b["size"] for b in pdf_blocks)
body_size = size_counts.most_common(1)[0][0]
print(f"Detected body font size: {body_size}")
print(f"Total text spans extracted: {len(pdf_blocks)}")


Detected body font size: 10.0
Total text spans extracted: 45731


## Step 3 — Semantic and Token-Aware Chunking

Raw PDF text must be broken into training-ready chunks that are:
- **Semantically coherent** — grouped around section headers so each chunk covers one topic.
- **Right-sized** — between 100 and 512 tokens to avoid padding waste and truncation.

Two classes handle this:

### `SemanticChunker`
Groups consecutive text spans into chunks separated by section headers. A span is classified as a header if it is:
- Larger than the body font size by more than 1 pt, **or**
- Bold (`flags & 1`), **or**
- Matches a numbered-section pattern (e.g. `7.1 Process`).

### `SmartChunker` (extends `SemanticChunker`)
Applies token-count constraints on top of semantic grouping:
- **Too short** (< 100 tokens) → merged into the previous chunk.
- **Too long** (> 512 tokens) → split on sentence/paragraph boundaries; hard-split by word if a single sentence still exceeds the limit.

In [37]:
from transformers import AutoTokenizer

# Load a tokenizer — swap in your target model name if needed
TOKENIZER_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


class SemanticChunker:
    """
    Groups text spans into semantic chunks using font size and bold flags
    to detect section headers — suited for structured spec documents like BPMN.
    """
    SECTION_RE = re.compile(r'^\d+(\.\d+)*\s+\w+')  # e.g. "7.1 Process"

    def __init__(self, blocks, body_size, size_threshold=1.0):
        self.blocks = blocks
        self.body_size = body_size
        self.size_threshold = size_threshold
        self.chunks = self._chunk_blocks()

    def _is_header(self, block):
        is_larger = block["size"] > self.body_size + self.size_threshold
        is_bold = bool(block["flags"] & 1)
        is_section = bool(self.SECTION_RE.match(block["text"]))
        return is_larger or is_bold or is_section

    def _chunk_blocks(self):
        chunks = []
        current = {"header": "", "value": ""}
        for block in self.blocks:
            if self._is_header(block):
                if current["header"] or current["value"].strip():
                    chunks.append(dict(current))
                current = {"header": block["text"], "value": ""}
            else:
                current["value"] += block["text"] + " "
        if current["header"] or current["value"].strip():
            chunks.append(current)
        return [c for c in chunks if c["value"].strip()]

    def get_merged_chunks(self):
        return [
            f"{c['header']} - {c['value'].strip()}" if c["header"] else c["value"].strip()
            for c in self.chunks
        ]


class SmartChunker(SemanticChunker):
    """Extends SemanticChunker with token-based min/max size enforcement."""

    def __init__(self, blocks, body_size, min_tokens=100, max_tokens=512):
        super().__init__(blocks, body_size)
        self.min_tokens = min_tokens
        self.max_tokens = max_tokens
        self.smart_chunks = self._adjust_chunks()

    def _split_by_tokens(self, text):
        """Split text into sub-chunks each <= max_tokens, on sentence boundaries."""
        splits = re.split(r'(?<=[.!?])\s+|\n\n|\n', text)
        sub_chunks = []
        current = ""
        for s in splits:
            candidate = (current + " " + s).strip() if current else s
            if count_tokens(candidate) <= self.max_tokens:
                current = candidate
            else:
                if current:
                    sub_chunks.append(current.strip())
                # If a single sentence is still too long, hard-split by words
                if count_tokens(s) > self.max_tokens:
                    words = s.split()
                    buf = ""
                    for w in words:
                        if count_tokens(buf + " " + w) <= self.max_tokens:
                            buf = (buf + " " + w).strip()
                        else:
                            if buf:
                                sub_chunks.append(buf)
                            buf = w
                    if buf:
                        sub_chunks.append(buf)
                    current = ""
                else:
                    current = s
        if current:
            sub_chunks.append(current.strip())
        return sub_chunks

    def _adjust_chunks(self):
        adjusted = []
        for chunk in self.get_merged_chunks():
            chunk = chunk.strip()
            if not chunk:
                continue
            tokens = count_tokens(chunk)
            if tokens < self.min_tokens:
                # Merge into previous chunk if it won't exceed max
                if adjusted and count_tokens(adjusted[-1]) + tokens <= self.max_tokens:
                    adjusted[-1] += " " + chunk
                else:
                    adjusted.append(chunk)
            elif tokens > self.max_tokens:
                adjusted.extend(self._split_by_tokens(chunk))
            else:
                adjusted.append(chunk)
        return adjusted


# Usage with font-aware structure (100–512 tokens per chunk)
smart_chunker = SmartChunker(pdf_blocks, body_size, min_tokens=100, max_tokens=512)
print(f"Total smart chunks: {len(smart_chunker.smart_chunks)}")
print("\nFirst 3 chunks:")
for i, c in enumerate(smart_chunker.smart_chunks[:3]):
    print(f"\n--- Chunk {i+1} (tokens={count_tokens(c)}) ---\n{c}")


Total smart chunks: 585

First 3 chunks:

--- Chunk 1 (tokens=17) ---
Business Process Model and Notation (BPMN), v2.0 xxiii

--- Chunk 2 (tokens=251) ---
About the Object Management Group - OMG Founded in 1989, the Object Management Group, Inc. (OMG) is an open membership, not-for-profit computer industry standards consortium that produces and maintains computer industry specifications for interoperable, portable and reusable enterprise applications in distributed, heterogeneous environments. Membership includes Information Technology vendors, end users, government agencies and academia. OMG member companies write, adopt, and maintain its specifications following a mature, open process. OMG's specifications implement the Model Driven Architecture® (MDA®), maximizing ROI through a full-lifecycle approach to enterprise integration that covers multiple operating systems, programming languages, middleware and networking infrastructures, and software development environments. OMG's specifi

## Step 4 — Inspect Token Distribution of the Chunks

Before training, it is important to verify that the chunks are well-distributed in terms of token count. This cell prints summary statistics including:
- **Total number of chunks** — the size of the training corpus.
- **Min / Max / Average token counts** — the spread of chunk sizes.
- **Chunks below 100 tokens** — potential information-sparse examples.
- **Chunks above 512 tokens** — examples that would be truncated and should be zero after the SmartChunker runs.

In [38]:
data = [{"text": p} for p in smart_chunker.smart_chunks]

# Token distribution of chunks
token_counts = [count_tokens(p) for p in smart_chunker.smart_chunks]
print(f"Total chunks:       {len(token_counts)}")
print(f"Min tokens:         {min(token_counts)}")
print(f"Max tokens:         {max(token_counts)}")
print(f"Avg tokens:         {sum(token_counts)//len(token_counts)}")
print(f"Chunks < 100 tok:   {sum(1 for t in token_counts if t < 100)}")
print(f"Chunks > 512 tok:   {sum(1 for t in token_counts if t > 512)}")


Total chunks:       585
Min tokens:         2
Max tokens:         512
Avg tokens:         360
Chunks < 100 tok:   39
Chunks > 512 tok:   0


## Step 5 — Build a Hugging Face Dataset

Convert the list of text chunks into a Hugging Face `Dataset` object. Each example has a single field `"text"` containing one chunk. This standardised format is required by the Hugging Face `Trainer` API used in the training step below.

In [39]:
from datasets import Dataset, load_dataset
dataset = Dataset.from_list(data)

dataset

Dataset({
    features: ['text'],
    num_rows: 585
})

## Step 6 — Load the Base Model and Tokenizer

Load the **TinyLlama-1.1B** base model and its associated tokenizer from Hugging Face Hub. TinyLlama is a compact 1.1-billion-parameter causal language model, making it suitable for experimentation on consumer-grade hardware.

**Tokenizer configuration:** TinyLlama does not define a `pad_token` by default (because it is a causal model trained without padding). We set it equal to `eos_token` so that the data collator can batch examples of different lengths.

In [41]:
#Model


# model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "meta-llama/Llama-3.1-8B"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

tokenizer = AutoTokenizer.from_pretrained(model_name)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Step 7 — Define the Tokenization Function

The `tokenize_fn` function converts each raw text chunk into model-ready token IDs. Key parameters:
- **`truncation=True`** — clips chunks longer than `max_length` tokens (safety guard).
- **`padding="max_length"`** — pads shorter chunks to a uniform length of 512 tokens, required for batched training.
- **`labels`** — set equal to `input_ids` so the model trains to predict every token (standard causal language modelling objective). The `Trainer` automatically shifts labels by one position internally.

In [42]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

## Step 8 — Tokenize the Dataset

Apply `tokenize_fn` to the entire dataset using batched processing. The original `"text"` column is removed after tokenization because the `Trainer` only needs the token ID and label tensors. The output `tokenized` dataset contains `input_ids`, `attention_mask`, and `labels` columns.

In [44]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

tokenized

Map: 100%|██████████| 585/585 [00:00<00:00, 2804.79 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 585
})

## Step 9 — Load the Causal Language Model

Load the full `TinyLlama-1.1B` model weights for causal language modelling. At this point the model has no LoRA adapters and all parameters are trainable (though LoRA will freeze the base weights in the next step). Loading onto CPU is acceptable because the LoRA adapter represents only a small fraction of total parameters and training on a single consumer GPU is feasible.

In [45]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1893.21it/s]


## Step 10 — Configure the LoRA Adapter

**LoRA (Low-Rank Adaptation)** injects small trainable matrices into selected attention layers, allowing the model to be fine-tuned with far fewer parameters than full fine-tuning.

Key configuration choices:
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `r` | 8 | Rank of the update matrices. Higher rank = more capacity, but more memory. |
| `lora_alpha` | 16 | Scaling factor (effective learning rate scales as `lora_alpha / r = 2`). |
| `target_modules` | `q_proj`, `v_proj` | Attention query and value projections — smallest set that gives meaningful adaptation. |
| `lora_dropout` | 0.05 | Light regularisation to prevent overfitting during domain adaptation. |
| `bias` | `"none"` | Bias terms are not trained, keeping the adapter small. |

In [47]:
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

## Step 11 — Apply LoRA Adapter to the Model

Wrap the base model with `get_peft_model`, which:
1. Freezes all original model parameters (they become non-trainable).
2. Inserts the low-rank LoRA matrices into the specified attention layers.
3. Returns a `PeftModel` where only the LoRA parameters (~1–5 M parameters) are updated during training, significantly reducing memory and compute requirements.

In [48]:
# save adapter
non_inst_model_lora = get_peft_model(model, lora_config)

## Step 12 — Set Training Hyperparameters

Configure the Hugging Face `TrainingArguments` for the LoRA fine-tuning run.

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `num_train_epochs` | 5 | Enough passes to adapt to the domain without overfitting on the spec text. |
| `per_device_train_batch_size` | 1 | Keeps GPU memory usage low. Effective batch = 1 × 8 = 8 (via gradient accumulation). |
| `gradient_accumulation_steps` | 8 | Simulates a larger batch without storing all gradients simultaneously. |
| `learning_rate` | 2e-4 | Standard LoRA learning rate — higher than full fine-tuning because only adapters are updated. |
| `fp16` | `True` | Mixed-precision training to reduce memory footprint and speed up computation. |
| `save_total_limit` | 1 | Keeps only the latest checkpoint to save disk space. |
| `report_to` | `"none"` | Disables all experiment tracking integrations (Weights & Biases, TensorBoard). |

In [49]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

## Step 13 — Initialise the Trainer

Create a Hugging Face `Trainer` instance, passing:
- The LoRA-wrapped model (`non_inst_model_lora`).
- The training arguments defined above.
- The tokenized dataset.

No separate `eval_dataset` is provided here because this stage focuses purely on domain adaptation (loss reduction), not task-specific evaluation. Evaluation will be done manually via generation in the test cell.

In [50]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

## Step 14 — Train the Model

Launch the training loop. The `Trainer` iterates over the tokenized dataset for the configured number of epochs, computing causal language modelling loss and updating only the LoRA adapter parameters.

The trained adapter checkpoint is saved to `./tinyllama-lora/` and will be used as the foundation for instruction fine-tuning in **Notebook 2.0**.

> **Expected duration:** Several minutes to ~1 hour depending on hardware (GPU recommended). Training on CPU is possible but significantly slower.

In [51]:
trainer.train()

c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
20,3.436756
40,1.591585
60,1.470852
80,1.324746
100,1.488201
120,1.412755
140,1.392540
160,1.390318
180,1.326294
200,1.335845


TrainOutput(global_step=370, training_loss=1.482767527812236, metrics={'train_runtime': 122303.04, 'train_samples_per_second': 0.024, 'train_steps_per_second': 0.003, 'total_flos': 9305835857510400.0, 'train_loss': 1.482767527812236, 'epoch': 5.0})

## Step 15 — Test the Domain-Adapted Model (Inference)

Load the saved LoRA adapter from `checkpoint-370` on top of the base TinyLlama model, and run a sample generation to verify that the model has learned BPMN-specific language.

**Generation parameters:**
- `max_new_tokens=100` — generate up to 100 tokens.
- `temperature=0.8`, `top_p=0.9` — nucleus sampling for diverse but coherent output.
- `repetition_penalty=1.1` — discourages the model from repeating the same phrases.

> **What to expect:** The model should continue the BPMN-specific prompt with contextually relevant content. Unlike the base model, it should demonstrate knowledge of BPMN terminology (e.g. escalation events, activities, boundaries). This is not yet instruction-following capability — that is added in Notebook 2.0.

In [71]:
import torch
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = "./tinyllama-lora/checkpoint-370"

base_model = AutoModelForCausalLM.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, model_path)
model = model.to(device)
model.eval()

prompt = "Escalation: Attached to Activity boundary"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4980.89it/s]
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

Escalation: Attached to Activity boundary or Participant boundary. If it is attached to the boundary, it shall be labeled as an escalation label (see page 30).
